# Employee Attrition & Financial Risk Analytics
## Notebook 3 of 3 — Model Explainability (SHAP) & Power BI Dashboard Exports

**Goal of this notebook:** explain *why* the model flags certain employees as high risk using
SHAP, then package clean, presentation-ready exports for Power BI.

**Inputs required** (all produced by earlier notebooks — this notebook runs from a fresh kernel
with no in-memory dependency on Notebooks 1 or 2):
- `models/logistic_regression_model.pkl`, `models/scaler.pkl`
- `data/X_train.csv`, `data/X_test.csv`, `data/y_train.csv`, `data/y_test.csv`
- `results/predictions_with_financial_risk.csv`
- `data/employee_attrition_clean.csv`

**Outputs of this notebook:**
- `visualizations/SHAP_Beeswarm.png`, `SHAP_bar.png`, `SHAP_Individual_Waterfall.png`
- `outputs/powerbi_shap_dashboard.csv`
- `outputs/powerbi_executive_dashboard.csv`
- `outputs/powerbi_financial_dashboard.csv`


## 1. Setup — Reload All Artifacts from Notebooks 1 & 2

This is the dependency hand-off point. Everything needed is reloaded from disk rather than
assumed to still be in memory, which is what makes this notebook independently runnable.

The **fitted** scaler is used with `.transform()` (never `.fit_transform()`) so the scaled
features here are identical to what the model was trained and evaluated on in Notebook 2 — this
is essential for SHAP values to be meaningful.


In [ ]:
import os
import joblib
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("visualizations", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

pd.set_option('display.max_columns', None)

# Model + fitted scaler from Notebook 2
model = joblib.load('models/logistic_regression_model.pkl')
scaler = joblib.load('models/scaler.pkl')

# Raw (unscaled) train/test splits from Notebook 2
X_train = pd.read_csv('data/X_train.csv', index_col=0)
X_test = pd.read_csv('data/X_test.csv', index_col=0)
y_train = pd.read_csv('data/y_train.csv', index_col=0).squeeze("columns")
y_test = pd.read_csv('data/y_test.csv', index_col=0).squeeze("columns")

# Predictions + financial risk dataset from Notebook 2
predictions = pd.read_csv('results/predictions_with_financial_risk.csv')

# Cleaned dataset from Notebook 1 (needed for the BI export section)
df_clean = pd.read_csv('data/employee_attrition_clean.csv')
df_clean['SalaryGroup'] = pd.Categorical(
    df_clean['SalaryGroup'],
    categories=['Low', 'Medium', 'High', 'Very High'],
    ordered=True
)
df_clean['AgeGroup'] = pd.Categorical(
    df_clean['AgeGroup'],
    categories=['18-25', '26-35', '36-45', '46-60'],
    ordered=True
)

print("Model, scaler, splits, predictions, and cleaned dataset loaded successfully.")


In [ ]:
predictions.info()


In [ ]:
predictions.head()


## 2. SHAP Explainability

[SHAP](https://shap.readthedocs.io/) (SHapley Additive exPlanations) attributes each prediction
to the individual features that drove it — this is what turns the model from a black box into
something an HR stakeholder can actually act on ("this employee is high risk *because of X, Y,
Z*").


In [ ]:
import shap


In [ ]:
print(shap.__version__)


Reproducing the exact scaled features from Notebook 2 using the loaded scaler.


In [ ]:
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns = X_test.columns,
    index = X_test.index
)


In [ ]:
explainer = shap.Explainer(model,X_train_scaled_df)


In [ ]:
shap_values = explainer(X_test_scaled_df)


In [ ]:
print(type(shap_values))
print(shap_values.shape)


### 2.1 Global Feature Importance


In [ ]:
shap.plots.beeswarm(shap_values, max_display = 20, show=False)

plt.savefig("visualizations/SHAP_Beeswarm.png", dpi = 300,bbox_inches="tight")

plt.show()


In [ ]:
shap.plots.bar(shap_values,max_display=15,show=False)

plt.savefig("visualizations/SHAP_bar.png", dpi = 300,bbox_inches="tight")

plt.show()


**Finding:** `OverTime`, `MaritalStatus`, and job-involvement-related features are consistently
among the strongest drivers of predicted attrition risk — consistent with the crosstab findings
from Notebook 1.


### 2.2 Individual Employee Explanation

A single employee's prediction, broken down feature-by-feature.


In [ ]:
employee_index = 0

shap.plots.waterfall(shap_values[employee_index], max_display = 15, show=False)

plt.savefig("visualizations/SHAP_Individual_Waterfall.png",dpi=300,bbox_inches='tight')

plt.show()


In [ ]:
print("Actual Attrition: ",y_test.iloc[employee_index])
print("Predicted Attrition: ",model.predict(X_test_scaled_df.iloc[[employee_index]],)[0])
print("Probability of Leaving: ",
      model.predict_proba
       (X_test_scaled_df.iloc[[employee_index]])[0][1]
      )


## 3. Building a Per-Employee SHAP Dashboard

Converting the SHAP values into a long-format table (`shap_long`) that a BI tool can filter and
aggregate — one row per employee/feature combination, with the SHAP value and its direction
(increases vs. reduces risk).


In [ ]:
print(shap_values.values.shape)


In [ ]:
shap_df = pd.DataFrame(
    shap_values.values,
    columns=X_test_scaled_df.columns,
    index=X_test_scaled_df.index
)

shap_df.head()


In [ ]:
print(shap_df.shape)


In [ ]:
shap_long = (
    shap_df
    .reset_index(names='EmployeeID')
    .melt(
        id_vars = 'EmployeeID',
        var_name = 'Feature',
        value_name = 'SHAP_Value'
    )
)

shap_long.head()


In [ ]:
shap_long["Impact"] = shap_long["SHAP_Value"].apply(
    lambda x: "Increases Risk" if x>0 else "Reduce Risk"
)

shap_long.head()


In [ ]:
print(shap_long.shape)


In [ ]:
predictions.columns.tolist()


In [ ]:
employee_summary = predictions[[
    "ActualAttrition",
    "PredictedAttrition",
    "Probability_of_Leaving",
    "FinancialRisk",
    "RiskCategory"
]
].copy()

employee_summary["EmployeeID"] = employee_summary.index
employee_summary.head()



In [ ]:
shap_dashboard = shap_long.merge(
    employee_summary,
    on="EmployeeID",
    how="left"
)

shap_dashboard.shape
shap_dashboard.head()


### 3.1 Top Risk Drivers per Employee

For each employee, the top 5 features increasing their risk and top 5 reducing it, ranked by
magnitude — this is the table that actually gets consumed in a Power BI "why is this employee
at risk" view.


In [ ]:
top_shap_rows = []

for employee in shap_df.index:
  temp = pd.DataFrame({
      "Feature": shap_df.columns,
      "SHAP": shap_df.loc[employee].values
  })

  top_positive = (
      temp[temp["SHAP"]>0]
      .sort_values("SHAP", ascending = False)
      .head(5)
  )

  top_negative = (
      temp[temp["SHAP"]<0]
      .sort_values("SHAP")
      .head(5)
  )

  top_negative["Impact"] = "Reduces Risk"
  top_positive["Impact"] = "Increases Risk"

  final = pd.concat([top_positive, top_negative])

  final["Employee"] = employee
  top_shap_rows.append(final)



In [ ]:
top_shap_df = pd.concat(top_shap_rows,ignore_index=True)
top_shap_df


In [ ]:
top_shap_df.shape


In [ ]:
print(shap_df.index[:10])


In [ ]:
top_shap_df.rename(columns={
    "Employee":"EmployeeID",
    "SHAP":"SHAP_Value"
},inplace=True)


In [ ]:
top_shap_df["Rank"] = (
    top_shap_df
    .assign(AbsSHAP=top_shap_df["SHAP_Value"].abs())
    .groupby(["EmployeeID","Impact"])
    ["AbsSHAP"].rank(method = 'first', ascending = False)
    .astype(int)
)

top_shap_df


In [ ]:
top_shap_dashboard = top_shap_df.merge(
    employee_summary,
    on="EmployeeID",
    how='left'
)

top_shap_dashboard.shape


In [ ]:
top_shap_dashboard


One-hot-encoded feature names (e.g. `OverTime_Yes`) are mapped back to plain, business-friendly
labels for the dashboard.


In [ ]:
feature_mapping = {
    "OverTime_Yes": "Overtime",
    "BusinessTravel_Travel_Frequently": "Travels Frequently",
    "BusinessTravel_Travel_Rarely": "Travels Rarely",
    "Department_Sales": "Sales Department",
    "Department_Research & Development": "R&D Department",
    "JobRole_Sales Executive": "Sales Executive",
    "JobRole_Research Scientist": "Research Scientist",
    "JobRole_Laboratory Technician": "Laboratory Technician",
    "JobRole_Manager": "Manager",
    "JobRole_Manufacturing Director": "Manufacturing Director",
    "JobRole_Human Resources": "Human Resources",
    "EducationField_Life Sciences": "Life Sciences",
    "EducationField_Medical": "Medical",
    "EducationField_Marketing": "Marketing",
    "EducationField_Technical Degree": "Technical Degree",
    "EducationField_Other": "Other Education",
    "Gender_Male": "Male",
    "MaritalStatus_Married": "Married",
    "MaritalStatus_Single": "Single"
}


In [ ]:
top_shap_dashboard["Feature"] = (
    top_shap_dashboard["Feature"].
    replace(feature_mapping)
)


In [ ]:
top_shap_dashboard


In [ ]:
top_shap_dashboard["SHAP_Value"] = (
    top_shap_dashboard["SHAP_Value"].round(3)
)

top_shap_dashboard


In [ ]:
top_shap_dashboard.to_csv(
    "outputs/powerbi_shap_dashboard.csv", index=False
)

print("PowerBI SHAP Dataset saved successfully")


## 4. Power BI Dashboard Exports

Two more presentation-ready extracts: an **Executive Dashboard** (high-level attrition view) and
a **Financial Dashboard** (risk-focused view with anonymized/synthetic employee names for
privacy).


### 4.1 Executive Dashboard

In [ ]:
df_clean.head


In [ ]:
executive_dashboard = df_clean[
    [
    "Department",
    "JobRole",
    "AgeGroup",
    "SalaryGroup",
    "EstimatedMonthlySalary",
    "BusinessTravel",
    "OverTime",
    "Attrition",
    "Gender"
    ]
].copy()

executive_dashboard.head(2)


In [ ]:
executive_dashboard["BusinessTravel"] = executive_dashboard["BusinessTravel"].replace(
    {
        "Travel_Rarely" : "Travel Rarely",
        "Travel_Frequently" : "Travel Frequently",
    }
)


In [ ]:
executive_dashboard.head()


In [ ]:
executive_dashboard.to_csv(
    "outputs/powerbi_executive_dashboard.csv", index=False
)


### 4.2 Financial Dashboard

In [ ]:
predictions.columns.tolist()


In [ ]:
financial_dashboard = predictions[
    [
    "Department",
    "JobRole",
    "AgeGroup",
    "SalaryGroup",
    "BusinessTravel",
    "OverTime",
    "PredictedAttrition",
    "Probability_of_Leaving",
    "ReplacementCost",
    "FinancialRisk",
    "RiskCategory"
    ]
].copy()


In [ ]:
financial_dashboard['BusinessTravel'] = financial_dashboard['BusinessTravel'].replace(
    {
    "Travel_Rarely": "Travel Rarely",
    "Travel_Frequently": "Travel Frequently",
    "Non-Travel": "Non-Travel"
    }
)


In [ ]:
financial_dashboard['EmployeeID'] = financial_dashboard.index


In [ ]:
financial_dashboard['Probability_of_Leaving'] = (financial_dashboard['Probability_of_Leaving'].round(3))


Employee names in the source dataset are not real (IBM's dataset is anonymized/synthetic to
begin with) — synthetic Indian names are generated purely to make the exported dashboard easier
to read for a non-technical audience, not to represent real individuals.


In [ ]:
!pip install faker
from faker import Faker
fake = Faker('en_IN')


In [ ]:
employee_names = {}

for emp_id in financial_dashboard['EmployeeID'].unique():
  employee_names[emp_id] = fake.name()

financial_dashboard['EmployeeName'] = financial_dashboard['EmployeeID'].map(employee_names)


In [ ]:
financial_dashboard[['EmployeeName','EmployeeID']].head()


In [ ]:
financial_dashboard.head(3)


In [ ]:
financial_dashboard.to_csv(
    "outputs/powerbi_financial_dashboard.csv", index=False
)

print("Financial Dashboard Saved Successfully")


---
**Project complete.** Three outputs are now available in `outputs/` for Power BI: the SHAP
explainability dashboard, the executive dashboard, and the financial risk dashboard — each
traceable back through `results/predictions_with_financial_risk.csv` and
`data/employee_attrition_clean.csv` to the raw source data explored in Notebook 1.
